Собрать один набор данных двумя способами: скриптом (requests + BeautifulSoup+XPath) и workflow в Octoparse (n8n,	Zapier и т.п.), сравнить трудозатраты и качество. Ожидаемый результат: два CSV-набора по одному источнику и сравнительная таблица «код vs no-code»

1. Выбрать сайт с которого можно произвести сбор данных. Например, 	https://books.toscrape.com/ - ссылка на сайт для scraping.
Использовать язык запросов XPath!
Напишите скрипт на Python для автоматического сбора информации с интернет-магазина. Скрипт должен загружать HTML-код страницы по заданному URL и извлекать данные обо всех доступных на ней товарах.
Каждый товар на сайте оформлен в виде блока с тегом <article>. 
Из каждой такой карточки необходимо извлечь:
- Название товара (содержится в атрибуте title ссылки внутри тега <h3>).
- Ссылку на изображение товара (содержится в атрибуте src тега <img>, который находится внутри блока с классом image_container).
- Цену товара (текст внутри тега <p> с классом price_color).

Требования к реализации:

Для загрузки страницы используйте библиотеку requests.
Для поиска и извлечения данных используйте библиотеку lxml (или html из lxml) и язык запросов XPath.
Выведите собранные данные (название, путь к картинке и цену) для каждого товара в консоль (в json?) в формате: 
Название - Путь_к_картинке - Цена.
	

In [ ]:
import time
import requests
import pandas as pd
from lxml import html

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
    )
}

TIMEOUT = 15
MAX_RETRIES = 3
RETRY_DELAY = 2
MAX_PAGES = 50

In [ ]:
def fetch(url: str) -> str | None:
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = requests.get(url, headers=HEADERS, timeout=TIMEOUT)

            if r.status_code == 200:
                r.encoding = r.apparent_encoding
                return r.text
            
            if r.status_code == 404:
                return None
            
            print(f"попытка {attempt}: HTTP {r.status_code}")
        except requests.RequestException as e:
            print(f"попытка {attempt}: {e}")

        if attempt < MAX_RETRIES:
            time.sleep(RETRY_DELAY * attempt)

    print(f"не удалось загрузить {url}")
    return None

# CSS-селекторы

Карточка товара: article.product_pod
- контейнер одного товара

Название: article.product_pod h3 a
- атрибут title

Картинка: .image_container img
- атрибут src

Цена: p.price_color
- текст внутри тега

In [41]:
all_books = []
page = 1

while page <= MAX_PAGES:
    page_url = f"https://books.toscrape.com/catalogue/page-{page}.html"
    print(f"Страница {page}: {page_url}")

    html_text = fetch(page_url)
    if html_text is None:
        print("страницы закончились")
        break

    tree = html.fromstring(html_text)
    cards = tree.xpath('//article[@class="product_pod"]')

    for card in cards:
        title = (card.xpath('.//h3/a/@title') or [""])[0].strip()

        img = (card.xpath('.//div[contains(@class,"image_container")]//img/@src') or [""])[0].strip()
        if img.startswith("../"):
            img = "https://books.toscrape.com/" + img[3:]
        elif img.startswith("/"):
            img = "https://books.toscrape.com" + img

        price_raw = (card.xpath('.//p[contains(@class,"price_color")]/text()') or [""])[0].strip()

        all_books.append({
            "title": title,
            "image_url": img,
            "price": price_raw,
        })

    print(f"всего собрано: {len(all_books)}")
    page += 1

print(f"\n Записей: {len(all_books)}")

Страница 1: https://books.toscrape.com/catalogue/page-1.html
всего собрано: 20
Страница 2: https://books.toscrape.com/catalogue/page-2.html
всего собрано: 40
Страница 3: https://books.toscrape.com/catalogue/page-3.html
всего собрано: 60
Страница 4: https://books.toscrape.com/catalogue/page-4.html
всего собрано: 80
Страница 5: https://books.toscrape.com/catalogue/page-5.html
всего собрано: 100
Страница 6: https://books.toscrape.com/catalogue/page-6.html
всего собрано: 120
Страница 7: https://books.toscrape.com/catalogue/page-7.html
всего собрано: 140
Страница 8: https://books.toscrape.com/catalogue/page-8.html
всего собрано: 160
Страница 9: https://books.toscrape.com/catalogue/page-9.html
всего собрано: 180
Страница 10: https://books.toscrape.com/catalogue/page-10.html
всего собрано: 200
Страница 11: https://books.toscrape.com/catalogue/page-11.html
всего собрано: 220
Страница 12: https://books.toscrape.com/catalogue/page-12.html
всего собрано: 240
Страница 13: https://books.toscrape.co

In [42]:
df = pd.DataFrame(all_books)

for col in df.columns:
    df[col] = df[col].astype(str).str.strip()

df["price"] = (
    df["price"]
    .str.replace("£", "", regex=False)
    .str.replace(",", ".", regex=False)
    .pipe(pd.to_numeric, errors="coerce")
)

df = df.replace({"": pd.NA})

before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
after = len(df)
print(f"Удалено дубликатов: {before - after}")

print(f"Итоговый датасет: {len(df)} строк")
print(f"Типы данных:\n{df.dtypes}")

Удалено дубликатов: 0
Итоговый датасет: 1000 строк
Типы данных:
title            str
image_url        str
price        float64
dtype: object


In [ ]:
OUTPUT_CSV = "books_clean.csv"

df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
print(f"Сохранено в {OUTPUT_CSV}")

Сохранено в books_clean.csv


In [46]:
import pandas as pd

df = pd.read_csv(OUTPUT_CSV)
df

,title,image_url,price
0,A Light in the Attic,https://books.toscrape.com/media/cache/2c/da/2...,51.77
1,Tipping the Velvet,https://books.toscrape.com/media/cache/26/0c/2...,53.74
2,Soumission,https://books.toscrape.com/media/cache/3e/ef/3...,50.10
3,Sharp Objects,https://books.toscrape.com/media/cache/32/51/3...,47.82
4,Sapiens: A Brief History of Humankind,https://books.toscrape.com/media/cache/be/a5/b...,54.23
...,...,...,...
995,Alice in Wonderland (Alice's Adventures in Won...,https://books.toscrape.com/media/cache/96/ee/9...,55.53
996,"Ajin: Demi-Human, Volume 1 (Ajin: Demi-Human #1)",https://books.toscrape.com/media/cache/09/7c/0...,57.06
997,A Spy's Devotion (The Regency Spies of London #1),https://books.toscrape.com/media/cache/1b/5f/1...,16.97
998,1st to Die (Women's Murder Club #1),https://books.toscrape.com/media/cache/2b/41/2...,53.98


# Результаты

### Сравнительный анализ программного и No-Code методов парсинга

| Критерий | Программная реализация (Python) | No-code реализация (Octoparse) |
| :--- | :--- | :--- |
| **Скорость и сложность настройки** | Требует написания кода, ручного поиска селекторов в DevTools и отладки скрипта. | Настраивается визуально за пару минут кликами мыши. Шаблон карточек распознается автоматически. |
| **Извлечение полей и data-атрибутов** | Гибкое извлечение любых элементов и атрибутов через метод. Ограничений по структуре кода нет. | Текст и ссылки берутся автоматически. Сбор специфических data-атрибутов требует ручной конфигурации тегов в расширенных настройках.. |
| **Пагинация (переход по страницам)** | Реализуется кодом: через цикл по номерам страниц в URL или сбор ссылок со стандартной кнопки «Вперед». | Настраивается автоматически функцией "Loop click next page" при клике на элемент пагинации. |
| **Обработка ошибок, Timeout и Retries** | Полный контроль над процессом. Таймауты и повторные попытки гибко задаются. | Настраивается чек-боксами в свойствах шага (опции *Timeout* и *Retry* в интерфейсе Workflow). |
| **Очистка данных и приведение типов** | Pandas автоматически приводит типы, удаляет дубликаты и ищет пропуски. | Базовая очистка (функция *Refine data* -> Trim/Replace). |
| **Статистика и аналитика выборки** | Можно смотреть число строк, превью первых записей и сразу можно начать аналитику данных. | Показывает только общий счетчик строк в процессе работы. |
| **Отказоустойчивость и обработка исключений** | Полный контроль | Валидация параметров через предустановленные свойства шагов (*Timeout*, *Retry*) в интерфейсе построения Workflow. |


Время работы

- Python - 52 сек
- Octapars - 2 мин 45 сек